# Fetch the 58 labeled + weak-labeled Effusion RSNA-Knee studies (in-kernel, no API listing)

Runs against the competition data mounted at `/kaggle/input/...` — no
`competition_list_files` paging, so no rate limiting. This is the SAME
mechanism (`find_competition_root`, the per-study EMPTY-flag progress
loop, the size sanity-check) that already worked for the original 58
ground-truth studies — extended here to ALSO pull a sample of the
weak-labeled Effusion studies (`qknee/artifacts/effusion_scaled_labels.csv`,
classifier-labeled, `needs_review`-filtered "confident" rows) from the
SAME competition mount, since those studies are just other `train.csv`
rows with a blank `Effusion` column — no separate Kaggle Dataset upload
needed, the images are already sitting in the same `train_series/` mount.

Writes both sets into the same `train_series/<StudyInstanceUID>/<SeriesInstanceUID>/<SOPInstanceUID>.dcm`
layout `RSNAKneeDataset` expects, under `/kaggle/working/train_series/`.

**Why 600, not 1,000, weak-labeled studies:** the 58 ground-truth studies
measured 0.845GB on disk (14.57 MB/study average). At that rate, 1,000
weak-labeled studies would add ~14.57GB — 15.4GB combined with the GT
set, ~77% of Kaggle's ~20GB `/kaggle/working` quota, leaving only ~4.6GB
of margin for notebook/package overhead. 600 studies instead adds ~8.7GB
(~9.6GB combined, ~48% of quota, ~10.4GB margin) — comfortably safe.
`N_WEAK_LABELED_SAMPLE` below is a single tunable constant if you want to
push it higher and accept the tighter margin; the pre-flight size-estimate
cell will warn before copying if the projected total looks unsafe.

After this runs, **Save Version** and pull the output down locally via
`kaggle kernels output <user>/<slug> -p train_series/` or the web UI's
Output tab — no need to touch the local repo's fetch script.

In [ ]:
from pathlib import Path
import shutil
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
OUT_ROOT = Path("/kaggle/working/train_series")
MAX_SLICES_PER_SERIES = 4  # matches every local _fetch_rsna_*.py script's cap
TARGET_COLUMNS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Safe default per the size math in the markdown cell above (58 GT studies
# measured 0.845GB -> 14.57 MB/study; 600 weak-labeled studies keeps the
# combined total at ~9.6GB, ~48% of the ~20GB /kaggle/working quota).
# Raise this if you want more studies and accept a tighter margin.
N_WEAK_LABELED_SAMPLE = 600
QUOTA_BYTES = 20 * 1e9
SAFE_QUOTA_FRACTION = 0.75  # warn if the projected total would exceed this fraction of quota

In [ ]:
def find_competition_root(input_root: Path) -> Path:
    """Locates the mounted competition data dir — the one with both
    train.csv and a train_series/ subdir — rather than hardcoding the
    input folder name, since Kaggle mounts a competition under its
    dataset slug, which doesn't always match the competition ref
    (`rsna-knee-abnormality-detection`).

    Checks each direct child of input_root, and also one level deeper
    under `<child>/competitions/*/`, since a competition-list mount nests
    the data there while a single-competition-attach mount puts it
    directly under the child — search both, don't assume either layout."""

    def has_data(path: Path) -> bool:
        return (path / "train.csv").exists() and (path / "train_series").is_dir()

    for candidate in sorted(input_root.iterdir()):
        if has_data(candidate):
            return candidate
        competitions_dir = candidate / "competitions"
        if competitions_dir.is_dir():
            for nested in sorted(competitions_dir.iterdir()):
                if has_data(nested):
                    return nested
    raise FileNotFoundError(
        f"no directory under {input_root} (including one level under any "
        "competitions/ subfolder) has both train.csv and train_series/ — "
        "did you attach the rsna-knee-abnormality-detection competition data?"
    )


COMPETITION_ROOT = find_competition_root(INPUT_ROOT)
print(f"competition data: {COMPETITION_ROOT}")

In [ ]:
# Same narrowing logic as scripts/_fetch_rsna_labeled_subset.py main() —
# only the source paths changed (mounted input, not the repo root).
train = pd.read_csv(COMPETITION_ROOT / "train.csv")
labeled = train.loc[train[TARGET_COLUMNS].notna().any(axis=1), "StudyInstanceUID"].astype(str)
gt_target_studies = set(labeled)
print(f"target: {len(gt_target_studies)} ground-truth labeled studies")

series_meta = pd.read_csv(COMPETITION_ROOT / "train_series.csv")
series_meta["StudyInstanceUID"] = series_meta["StudyInstanceUID"].astype(str)
series_meta["SeriesInstanceUID"] = series_meta["SeriesInstanceUID"].astype(str)

def build_study_series_planes(target_studies: set) -> dict:
    return {
        study_uid: dict(zip(group["SeriesInstanceUID"], group["Anatomical_Plane"]))
        for study_uid, group in series_meta[series_meta["StudyInstanceUID"].isin(target_studies)].groupby("StudyInstanceUID")
    }

gt_study_series_planes = build_study_series_planes(gt_target_studies)
total_known_series = sum(len(v) for v in gt_study_series_planes.values())
print(f"these studies have {total_known_series} known series total (across all planes)")

In [ ]:
# Local filesystem walk instead of api.competition_list_files paging —
# the whole reason this notebook exists: the mount already has every
# individual .dcm filename on disk, for free, no rate limit. Extracted
# into a function so the SAME EMPTY-flag/per-study-progress pattern that
# worked for the 58 ground-truth studies is reused verbatim for the
# weak-labeled sample, not re-implemented.
def copy_studies(target_studies: set, study_series_planes: dict, label: str) -> dict:
    n_files_total = 0
    n_studies_empty = 0
    n_series_missing = 0

    for study_uid in sorted(target_studies):
        known_series = study_series_planes.get(study_uid, {})
        study_src = COMPETITION_ROOT / "train_series" / study_uid
        n_files_this_study = 0
        n_series_this_study = 0

        for series_uid in known_series:
            series_src = study_src / series_uid
            if not series_src.is_dir():
                n_series_missing += 1
                continue

            sop_files = sorted(p.name for p in series_src.iterdir() if p.is_file())[:MAX_SLICES_PER_SERIES]
            if not sop_files:
                continue

            series_dst = OUT_ROOT / study_uid / series_uid
            series_dst.mkdir(parents=True, exist_ok=True)
            for sop_file in sop_files:
                dst_file = series_dst / sop_file
                if not dst_file.exists():
                    shutil.copyfile(series_src / sop_file, dst_file)
            n_files_this_study += len(sop_files)
            n_series_this_study += 1

        n_files_total += n_files_this_study
        flag = "" if n_files_this_study else "  <-- EMPTY, check this study"
        print(
            f"[{label}] {study_uid}: {n_series_this_study}/{len(known_series)} series, "
            f"{n_files_this_study} files copied{flag}"
        )
        if not n_files_this_study:
            n_studies_empty += 1

    print()
    print(
        f"[{label}] DONE: {len(target_studies) - n_studies_empty}/{len(target_studies)} studies with >=1 file, "
        f"{n_files_total} files copied, {n_series_missing} known series missing from disk, "
        f"output -> {OUT_ROOT}"
    )
    if n_studies_empty:
        print(f"[{label}] WARNING: {n_studies_empty} studies came up completely empty — inspect before downloading.")

    return {"n_files": n_files_total, "n_studies_empty": n_studies_empty, "n_series_missing": n_series_missing}

In [ ]:
# Copy the 58 ground-truth studies (unchanged behavior from before this
# notebook was extended — same function, same pattern).
gt_stats = copy_studies(gt_target_studies, gt_study_series_planes, label="ground_truth")

In [ ]:
# Weak-labeled Effusion sample: pre-selected locally (fixed seed=42, from
# the 3,723 "confident" rows in qknee/artifacts/effusion_scaled_labels.csv
# -- needs_review blank -- via pandas .sample(n=600, random_state=42)) so
# this notebook stays self-contained -- no separate Kaggle Dataset upload
# needed, since the images themselves are already in COMPETITION_ROOT
# (these are just other train.csv studies with a blank Effusion column).
import io

WEAK_LABELED_SAMPLE_CSV = """StudyInstanceUID,assigned_label
1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981,1.0
1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866,0.0
1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269,0.0
1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364,1.0
1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100,0.0
1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882,1.0
1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343,1.0
1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834,0.0
1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790,0.0
1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205,1.0
1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415,0.0
1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973,0.0
1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773,0.0
1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183,0.0
1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585,0.0
1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326,0.0
1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766,0.0
1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542,0.0
1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097,1.0
1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466,1.0
1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726,1.0
1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179,0.0
1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341,1.0
1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007,0.0
1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056,1.0
1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409,0.0
1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884,0.0
1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584,0.0
1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117,0.0
1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213,1.0
1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627,0.0
1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828,1.0
1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801,1.0
1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953,0.0
1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260,1.0
1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169,0.0
1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852,1.0
1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628,0.0
1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137,1.0
1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196,1.0
1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430,1.0
1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593,1.0
1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971,1.0
1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831,1.0
1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286,0.0
1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080,0.0
1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557,0.0
1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171,0.0
1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826,1.0
1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334,0.0
1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770,1.0
1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002,1.0
1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263,0.0
1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650,0.0
1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636,1.0
1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931,1.0
1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748,0.0
1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758,0.0
1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433,1.0
1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722,0.0
1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424,1.0
1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100,0.0
1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480,1.0
1.2.826.0.1.3680043.8.498.11006241238894862275805535376070111510,0.0
1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256,1.0
1.2.826.0.1.3680043.8.498.11043081829401721095803441219553837608,1.0
1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298,0.0
1.2.826.0.1.3680043.8.498.11060834081463491262807709963877480026,0.0
1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547,0.0
1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635,0.0
1.2.826.0.1.3680043.8.498.11100224438877567316062152508871079233,1.0
1.2.826.0.1.3680043.8.498.11110575656494405149601490027444484170,0.0
1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427,0.0
1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290,0.0
1.2.826.0.1.3680043.8.498.11136723251983227230447257874027263286,0.0
1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618,1.0
1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731,1.0
1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678,0.0
1.2.826.0.1.3680043.8.498.11226582925516370364253861721606580867,1.0
1.2.826.0.1.3680043.8.498.11260248089108026217929330361529065683,1.0
1.2.826.0.1.3680043.8.498.11275711698468745807758222802302685022,0.0
1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372,1.0
1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607,0.0
1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512,0.0
1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463,1.0
1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623,1.0
1.2.826.0.1.3680043.8.498.11448326350892044215419380818679679579,0.0
1.2.826.0.1.3680043.8.498.11451981256762378203455481812714370193,0.0
1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769,1.0
1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981,0.0
1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588,1.0
1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983,0.0
1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355,0.0
1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974,0.0
1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425,1.0
1.2.826.0.1.3680043.8.498.11590096889491409910425456896187293515,0.0
1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202,1.0
1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479,0.0
1.2.826.0.1.3680043.8.498.11610585947047502123824835405421129552,1.0
1.2.826.0.1.3680043.8.498.11625598825018422177242262569319579378,0.0
1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093,1.0
1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493,0.0
1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917,0.0
1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637,0.0
1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753,1.0
1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015,1.0
1.2.826.0.1.3680043.8.498.11803762390999881908101130422708589537,0.0
1.2.826.0.1.3680043.8.498.11881735311125899170335963782017197184,1.0
1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369,1.0
1.2.826.0.1.3680043.8.498.11911688823908554151914734938172012708,1.0
1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920,1.0
1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600,0.0
1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317,1.0
1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948,0.0
1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084,1.0
1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122,0.0
1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179,0.0
1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267,0.0
1.2.826.0.1.3680043.8.498.12165893434332227221101266719291133169,0.0
1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217,0.0
1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986,0.0
1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249,0.0
1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199,1.0
1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573,0.0
1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033,0.0
1.2.826.0.1.3680043.8.498.12365033663152692616611548717152384269,0.0
1.2.826.0.1.3680043.8.498.12381909868368799280462187344958674397,1.0
1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696,1.0
1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831,0.0
1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080,1.0
1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699,0.0
1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136,1.0
1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727,1.0
1.2.826.0.1.3680043.8.498.12563940497360871791097093821926086616,0.0
1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184,0.0
1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842,1.0
1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718,1.0
1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635,0.0
1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625,1.0
1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405,0.0
1.2.826.0.1.3680043.8.498.12725832336445568031849006413563892122,0.0
1.2.826.0.1.3680043.8.498.12731497178056855493473947157407264492,1.0
1.2.826.0.1.3680043.8.498.12771448321084229688831202344007889535,0.0
1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180,1.0
1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802,0.0
1.2.826.0.1.3680043.8.498.12790542912560396272981117250277568532,1.0
1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010,1.0
1.2.826.0.1.3680043.8.498.12806792356922865929896103306611295151,0.0
1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174,0.0
1.2.826.0.1.3680043.8.498.12820051821104182459960036809430047569,0.0
1.2.826.0.1.3680043.8.498.13004048880335015585285100185910148808,0.0
1.2.826.0.1.3680043.8.498.13028226019192925363616760332442456953,1.0
1.2.826.0.1.3680043.8.498.13034169434311535671044673510644711229,1.0
1.2.826.0.1.3680043.8.498.13069081594164738319108129224400751974,1.0
1.2.826.0.1.3680043.8.498.13078551660783737720660607567713996703,1.0
1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987,1.0
1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590,1.0
1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258,1.0
1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937,0.0
1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501,1.0
1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604,0.0
1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737,0.0
1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680,1.0
1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636,0.0
1.2.826.0.1.3680043.8.498.13660439785395658826946769630921591235,0.0
1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847,1.0
1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838,0.0
1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895,0.0
1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166,0.0
1.2.826.0.1.3680043.8.498.14511862461248267537051973802039959211,1.0
1.2.826.0.1.3680043.8.498.15326951535563258414956521742264374802,1.0
1.2.826.0.1.3680043.8.498.15522600887743817464976568872439594397,0.0
1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840,0.0
1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081,0.0
1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921,0.0
1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943,0.0
1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358,1.0
1.2.826.0.1.3680043.8.498.16397020531009156965821242117417547331,0.0
1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835,1.0
1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849,1.0
1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645,1.0
1.2.826.0.1.3680043.8.498.17265093113586420061761042467476063876,1.0
1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509,1.0
1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823,0.0
1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519,1.0
1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854,1.0
1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730,1.0
1.2.826.0.1.3680043.8.498.18114567737638963627123562895002602285,0.0
1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080,1.0
1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569,1.0
1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908,0.0
1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006,0.0
1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566,0.0
1.2.826.0.1.3680043.8.498.18993960105596277135383396226525760874,0.0
1.2.826.0.1.3680043.8.498.19813267030974842239752491935957206955,1.0
1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391,1.0
1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886,0.0
1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640,1.0
1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537,1.0
1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849,1.0
1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871,0.0
1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044,0.0
1.2.826.0.1.3680043.8.498.21726746202429506433532477857404854137,0.0
1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104,1.0
1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537,1.0
1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643,0.0
1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060,1.0
1.2.826.0.1.3680043.8.498.22594249996322566888700984133307337128,1.0
1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503,1.0
1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038,0.0
1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720,0.0
1.2.826.0.1.3680043.8.498.25145588904873072866960435474159832047,0.0
1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709,0.0
1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319,1.0
1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077,0.0
1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863,0.0
1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104,0.0
1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487,0.0
1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878,1.0
1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736,1.0
1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208,1.0
1.2.826.0.1.3680043.8.498.26333849754345698117396119042432557445,0.0
1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387,1.0
1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512,0.0
1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004,1.0
1.2.826.0.1.3680043.8.498.27023892731021470476500135946317920771,0.0
1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211,1.0
1.2.826.0.1.3680043.8.498.27339999385861467858422021400023533486,0.0
1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482,0.0
1.2.826.0.1.3680043.8.498.27681651277422558344520227187021583924,0.0
1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421,0.0
1.2.826.0.1.3680043.8.498.27883371281368994280894952749865160316,1.0
1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935,0.0
1.2.826.0.1.3680043.8.498.27946605663086243142356841956045514227,1.0
1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325,0.0
1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028,0.0
1.2.826.0.1.3680043.8.498.28488216314048990109149780234417422948,0.0
1.2.826.0.1.3680043.8.498.28626044336302232656987963623687293705,1.0
1.2.826.0.1.3680043.8.498.28686498221359515429184385385077600687,0.0
1.2.826.0.1.3680043.8.498.29292398668403593100514493322849770181,1.0
1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001,1.0
1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464,1.0
1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856,0.0
1.2.826.0.1.3680043.8.498.29741673719687535294395668150749894623,0.0
1.2.826.0.1.3680043.8.498.29866827083314645739444078187995496049,1.0
1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854,1.0
1.2.826.0.1.3680043.8.498.30066410959464367576375502522994033142,1.0
1.2.826.0.1.3680043.8.498.30150613585152527569197215147135001922,0.0
1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073,0.0
1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260,0.0
1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989,0.0
1.2.826.0.1.3680043.8.498.30537361712821464927469618193893850465,0.0
1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701,0.0
1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793,1.0
1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777,0.0
1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486,1.0
1.2.826.0.1.3680043.8.498.31565705236407722278921437231164204914,1.0
1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778,1.0
1.2.826.0.1.3680043.8.498.32513010438338970418994141548516096399,1.0
1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933,1.0
1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664,1.0
1.2.826.0.1.3680043.8.498.33215387283623093392227819856970670071,0.0
1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510,0.0
1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521,1.0
1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550,1.0
1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314,0.0
1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755,1.0
1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830,0.0
1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167,1.0
1.2.826.0.1.3680043.8.498.33997019840409023169542161093833920630,0.0
1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835,0.0
1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542,1.0
1.2.826.0.1.3680043.8.498.34690162612328981644511154212357885455,1.0
1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776,1.0
1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720,0.0
1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384,0.0
1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565,0.0
1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991,1.0
1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868,1.0
1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554,1.0
1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903,1.0
1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597,0.0
1.2.826.0.1.3680043.8.498.37644754953609976838298110418046213654,0.0
1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606,0.0
1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267,0.0
1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200,0.0
1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065,1.0
1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060,1.0
1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405,1.0
1.2.826.0.1.3680043.8.498.39020025030013167602991688044932114354,1.0
1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161,1.0
1.2.826.0.1.3680043.8.498.39135770176150551756142329948474103964,1.0
1.2.826.0.1.3680043.8.498.39197976722851762826265579540678924584,1.0
1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611,0.0
1.2.826.0.1.3680043.8.498.39799597341675508044808282472428242917,0.0
1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085,1.0
1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230,0.0
1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818,1.0
1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201,1.0
1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754,0.0
1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116,0.0
1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496,1.0
1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615,0.0
1.2.826.0.1.3680043.8.498.41837648239243141231469586963862388683,0.0
1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406,0.0
1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633,0.0
1.2.826.0.1.3680043.8.498.42499092649567991851673654940784921876,0.0
1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276,0.0
1.2.826.0.1.3680043.8.498.43256282639827216310036033378453939750,1.0
1.2.826.0.1.3680043.8.498.43432499267536761263202158662502957684,1.0
1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343,1.0
1.2.826.0.1.3680043.8.498.43445351322695981909807931874796758490,1.0
1.2.826.0.1.3680043.8.498.43587191909319947256183198986671106743,1.0
1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739,1.0
1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208,1.0
1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727,0.0
1.2.826.0.1.3680043.8.498.44713330360073884105636085566230396530,0.0
1.2.826.0.1.3680043.8.498.44758766790156297849747618145458535803,0.0
1.2.826.0.1.3680043.8.498.45316453179223822297459860764883030492,1.0
1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641,1.0
1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159,0.0
1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775,1.0
1.2.826.0.1.3680043.8.498.45630051398709774279703407401383545642,0.0
1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426,1.0
1.2.826.0.1.3680043.8.498.45968957856865486437256472516713628614,0.0
1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479,1.0
1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552,1.0
1.2.826.0.1.3680043.8.498.46745172905542117506863575411125924241,1.0
1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541,0.0
1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684,1.0
1.2.826.0.1.3680043.8.498.47531034194300620953276025401018051246,1.0
1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981,0.0
1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166,0.0
1.2.826.0.1.3680043.8.498.48520238688122533624681796433258211236,1.0
1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236,1.0
1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142,1.0
1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619,1.0
1.2.826.0.1.3680043.8.498.49048739684287632681242054218807611262,0.0
1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662,0.0
1.2.826.0.1.3680043.8.498.49583348226932515919984811293816232513,1.0
1.2.826.0.1.3680043.8.498.49866252311077397175807461727885923657,0.0
1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125,0.0
1.2.826.0.1.3680043.8.498.49887758195164927372912584780941789021,1.0
1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913,0.0
1.2.826.0.1.3680043.8.498.50215409013618353346619153290049205718,0.0
1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052,0.0
1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679,0.0
1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069,1.0
1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367,0.0
1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742,1.0
1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545,1.0
1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257,0.0
1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099,0.0
1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608,0.0
1.2.826.0.1.3680043.8.498.52110163572817528199125320120779141456,0.0
1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152,0.0
1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159,1.0
1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711,1.0
1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716,1.0
1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832,0.0
1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146,0.0
1.2.826.0.1.3680043.8.498.53361558447470435467999344560545063264,1.0
1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092,0.0
1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424,0.0
1.2.826.0.1.3680043.8.498.54833384793167149128371137537527194757,1.0
1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864,0.0
1.2.826.0.1.3680043.8.498.55332521421041708438748125666498042763,1.0
1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886,0.0
1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246,1.0
1.2.826.0.1.3680043.8.498.55850385284038070407841702672827533728,0.0
1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738,0.0
1.2.826.0.1.3680043.8.498.55991293001848799160041967098772087659,1.0
1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391,0.0
1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616,1.0
1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923,0.0
1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006,0.0
1.2.826.0.1.3680043.8.498.56893426177755960410197244250599230068,1.0
1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369,1.0
1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597,1.0
1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044,1.0
1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210,0.0
1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163,1.0
1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419,1.0
1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292,1.0
1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917,1.0
1.2.826.0.1.3680043.8.498.58634494571833713296574414948455471101,0.0
1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656,0.0
1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744,0.0
1.2.826.0.1.3680043.8.498.58857477075095368361468658035369975572,1.0
1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714,0.0
1.2.826.0.1.3680043.8.498.59073122184069676919857712290159511305,1.0
1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768,1.0
1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420,0.0
1.2.826.0.1.3680043.8.498.60439571175176992075370053747045234796,0.0
1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474,1.0
1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592,0.0
1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106,0.0
1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417,0.0
1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569,0.0
1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103,1.0
1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682,1.0
1.2.826.0.1.3680043.8.498.63105151375349194807502162946823966410,1.0
1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540,1.0
1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519,0.0
1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473,0.0
1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334,0.0
1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453,1.0
1.2.826.0.1.3680043.8.498.64186424577324542519674308830472181373,1.0
1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223,0.0
1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525,0.0
1.2.826.0.1.3680043.8.498.65077085081642080527129904737510276643,1.0
1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252,1.0
1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325,0.0
1.2.826.0.1.3680043.8.498.65322445533737673246312151513913460816,0.0
1.2.826.0.1.3680043.8.498.65438121008780366054026567981577676554,0.0
1.2.826.0.1.3680043.8.498.65547889329285432771702699287910809921,1.0
1.2.826.0.1.3680043.8.498.66031142975529812579758652269360365398,1.0
1.2.826.0.1.3680043.8.498.66308195780676793917989368423009849637,1.0
1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709,0.0
1.2.826.0.1.3680043.8.498.66694519343321893230666089565893349853,0.0
1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525,1.0
1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269,0.0
1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713,1.0
1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159,0.0
1.2.826.0.1.3680043.8.498.67374476726034608591738910363595695123,1.0
1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451,0.0
1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748,0.0
1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378,0.0
1.2.826.0.1.3680043.8.498.67875980316053973061252844698884229168,0.0
1.2.826.0.1.3680043.8.498.67901792488494108577057563458320460313,1.0
1.2.826.0.1.3680043.8.498.67931702364673809781455566584550075686,0.0
1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666,0.0
1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623,0.0
1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606,0.0
1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335,1.0
1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471,0.0
1.2.826.0.1.3680043.8.498.69002590469755721401448924002989552840,1.0
1.2.826.0.1.3680043.8.498.69156800026549034619316278954442147703,1.0
1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455,0.0
1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198,1.0
1.2.826.0.1.3680043.8.498.69717507929047897681405659943474469018,1.0
1.2.826.0.1.3680043.8.498.70471859687546886677145171619638581757,1.0
1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827,0.0
1.2.826.0.1.3680043.8.498.70873330457307886117949106432465978541,1.0
1.2.826.0.1.3680043.8.498.70880546533848103776823598726224805522,1.0
1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266,1.0
1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315,1.0
1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565,1.0
1.2.826.0.1.3680043.8.498.71744523263023933066263506201344989825,0.0
1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637,1.0
1.2.826.0.1.3680043.8.498.72375103774239787721371778221019402834,0.0
1.2.826.0.1.3680043.8.498.72740499664060575780267947543695605585,0.0
1.2.826.0.1.3680043.8.498.72880034930990967339546757487847803284,0.0
1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398,0.0
1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751,1.0
1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320,1.0
1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465,1.0
1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951,0.0
1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186,0.0
1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871,0.0
1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385,1.0
1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661,0.0
1.2.826.0.1.3680043.8.498.74622031396885197580453304563088863451,0.0
1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738,1.0
1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303,0.0
1.2.826.0.1.3680043.8.498.75547515925579172661099437338637082554,0.0
1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542,0.0
1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662,0.0
1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386,1.0
1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992,1.0
1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903,1.0
1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636,0.0
1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695,1.0
1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219,1.0
1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809,0.0
1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971,0.0
1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727,1.0
1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146,0.0
1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239,0.0
1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586,0.0
1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568,0.0
1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052,1.0
1.2.826.0.1.3680043.8.498.78853931928960361455025737444267199231,0.0
1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949,1.0
1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942,1.0
1.2.826.0.1.3680043.8.498.79292862882980756453890778996691993351,0.0
1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807,0.0
1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138,0.0
1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754,1.0
1.2.826.0.1.3680043.8.498.80629565747112614845746475820152813307,0.0
1.2.826.0.1.3680043.8.498.80758336123908522542787716219939751704,1.0
1.2.826.0.1.3680043.8.498.81018911907787774214591206085583545057,0.0
1.2.826.0.1.3680043.8.498.81540134120507822899514611863725372563,1.0
1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166,0.0
1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661,0.0
1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236,1.0
1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172,0.0
1.2.826.0.1.3680043.8.498.82144602300666174450721045102799681426,0.0
1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974,1.0
1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134,1.0
1.2.826.0.1.3680043.8.498.82655005222804459605894403147145992714,1.0
1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914,0.0
1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538,1.0
1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925,1.0
1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239,1.0
1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332,1.0
1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462,0.0
1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057,0.0
1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991,0.0
1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475,0.0
1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396,0.0
1.2.826.0.1.3680043.8.498.85031127696558971105465423134104223548,0.0
1.2.826.0.1.3680043.8.498.85464150757448712711832350617010924327,1.0
1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590,1.0
1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647,1.0
1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800,0.0
1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607,0.0
1.2.826.0.1.3680043.8.498.85970222049966384156120176704985355996,1.0
1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827,0.0
1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144,0.0
1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972,0.0
1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251,1.0
1.2.826.0.1.3680043.8.498.87046380372347615219261898797043742937,0.0
1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868,0.0
1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188,1.0
1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015,0.0
1.2.826.0.1.3680043.8.498.88022456207467042544403752144674777523,1.0
1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905,0.0
1.2.826.0.1.3680043.8.498.88114918358665488269761388157480523350,0.0
1.2.826.0.1.3680043.8.498.88571985843649843469694556575325271560,1.0
1.2.826.0.1.3680043.8.498.88655485247489827200671636011965281184,1.0
1.2.826.0.1.3680043.8.498.88753415430913291809543943690942744997,0.0
1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256,1.0
1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294,0.0
1.2.826.0.1.3680043.8.498.89346975863478164334871204856160071946,0.0
1.2.826.0.1.3680043.8.498.89445342796957655182687412965056270903,1.0
1.2.826.0.1.3680043.8.498.90340114291852810896219650388952790164,0.0
1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056,1.0
1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353,1.0
1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085,1.0
1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680,0.0
1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506,0.0
1.2.826.0.1.3680043.8.498.91206973544724011503639050238118945528,1.0
1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637,0.0
1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999,1.0
1.2.826.0.1.3680043.8.498.91744384920314309659802891351055961910,1.0
1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273,0.0
1.2.826.0.1.3680043.8.498.92030581934838097812004394123949263069,0.0
1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745,0.0
1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494,1.0
1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881,1.0
1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506,0.0
1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780,0.0
1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369,0.0
1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055,0.0
1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449,0.0
1.2.826.0.1.3680043.8.498.93344237572402384894445814841725878706,1.0
1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308,0.0
1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303,1.0
1.2.826.0.1.3680043.8.498.94244885518758327449236878653096161475,0.0
1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914,1.0
1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516,1.0
1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046,0.0
1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449,0.0
1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472,1.0
1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537,1.0
1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946,0.0
1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963,1.0
1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981,0.0
1.2.826.0.1.3680043.8.498.95500234888498554736637315199668317948,0.0
1.2.826.0.1.3680043.8.498.95586477246200285088746818821808017494,0.0
1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522,0.0
1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460,0.0
1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951,0.0
1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466,0.0
1.2.826.0.1.3680043.8.498.96146854841734718524530740965296119916,0.0
1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217,1.0
1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186,1.0
1.2.826.0.1.3680043.8.498.96947237995559790704304892745226362883,1.0
1.2.826.0.1.3680043.8.498.97003157807630958179276831877591411623,0.0
1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254,0.0
1.2.826.0.1.3680043.8.498.97087958215554026148585108910609248523,0.0
1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509,0.0
1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499,0.0
1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086,0.0
1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265,0.0
1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720,1.0
1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132,1.0
1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526,0.0
1.2.826.0.1.3680043.8.498.98581537941208552065033680853629122764,1.0
1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070,1.0
1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659,0.0
1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766,0.0
1.2.826.0.1.3680043.8.498.99444110025206148999881196086066868280,1.0
1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135,1.0
1.2.826.0.1.3680043.8.498.99708702196778120583644711299950439036,0.0
1.2.826.0.1.3680043.8.498.99813269006669714758887909212386431934,1.0
1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820,1.0
1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290,0.0
1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209,1.0"""

weak_sample = pd.read_csv(io.StringIO(WEAK_LABELED_SAMPLE_CSV), dtype={"StudyInstanceUID": str})
weak_sample["assigned_label"] = weak_sample["assigned_label"].astype(int)
assert len(weak_sample) == N_WEAK_LABELED_SAMPLE, (
    f"embedded sample has {len(weak_sample)} rows, expected N_WEAK_LABELED_SAMPLE={N_WEAK_LABELED_SAMPLE} "
    "-- update the embedded CSV if you changed the sample size locally."
)
weak_target_studies = set(weak_sample["StudyInstanceUID"])
weak_study_series_planes = build_study_series_planes(weak_target_studies)
n_pos = int(weak_sample["assigned_label"].sum())
print(
    f"weak-labeled sample: {len(weak_target_studies)} studies "
    f"({n_pos} positive / {len(weak_target_studies) - n_pos} negative Effusion), "
    f"{sum(len(v) for v in weak_study_series_planes.values())} known series total"
)

# Pre-flight size estimate BEFORE copying -- extrapolates from the actual
# bytes just copied for the 58 GT studies (real measurement, not a guess),
# so this catches an oversized N_WEAK_LABELED_SAMPLE before it eats the
# /kaggle/working quota, not after.
gt_bytes_so_far = sum(p.stat().st_size for p in OUT_ROOT.rglob("*") if p.is_file())
bytes_per_gt_study = gt_bytes_so_far / max(len(gt_target_studies), 1)
projected_weak_bytes = bytes_per_gt_study * len(weak_target_studies)
projected_total_bytes = gt_bytes_so_far + projected_weak_bytes
print(
    f"pre-flight estimate: {bytes_per_gt_study/1e6:.2f} MB/study (from the {len(gt_target_studies)} "
    f"GT studies just copied) x {len(weak_target_studies)} weak-labeled studies "
    f"= ~{projected_weak_bytes/1e9:.2f} GB more -> projected total ~{projected_total_bytes/1e9:.2f} GB "
    f"({projected_total_bytes/QUOTA_BYTES*100:.0f}% of the ~{QUOTA_BYTES/1e9:.0f}GB quota)"
)
if projected_total_bytes > QUOTA_BYTES * SAFE_QUOTA_FRACTION:
    print(
        f"WARNING: projected total exceeds {SAFE_QUOTA_FRACTION*100:.0f}% of quota -- "
        "consider lowering N_WEAK_LABELED_SAMPLE (re-sample locally and re-embed) before running the next cell."
    )


In [ ]:
# Copy the weak-labeled sample -- same copy_studies function, same
# EMPTY-flag/per-study-progress pattern as the ground-truth copy above.
weak_stats = copy_studies(weak_target_studies, weak_study_series_planes, label="weak_labeled")

In [ ]:
# Sanity-check total output size against the ~20GB /kaggle/working quota
# before Save Version -- and the GT vs. weak-labeled breakdown, so it's
# obvious in the notebook output which pool each byte belongs to.
total_bytes = sum(p.stat().st_size for p in OUT_ROOT.rglob("*") if p.is_file())
total_files = sum(1 for _ in OUT_ROOT.rglob("*.dcm"))
print(f"total output size: {total_bytes / 1e9:.2f} GB across {total_files} files "
      f"({total_bytes/QUOTA_BYTES*100:.0f}% of the ~{QUOTA_BYTES/1e9:.0f}GB quota)")
print(f"  ground_truth: {len(gt_target_studies)} studies, {gt_stats['n_files']} files")
print(f"  weak_labeled: {len(weak_target_studies)} studies, {weak_stats['n_files']} files")